# The survey whose ages summed to a 40 000-digit number

Values read from a file are text until you say otherwise. Here is the bug,
reproduced in miniature.

In [1]:
import csv, io

raw = """id,age
1,23
2,41
3,35
4,29
"""
rows = list(csv.DictReader(io.StringIO(raw)))
print(rows[0], type(rows[0]["age"]))

{'id': '1', 'age': '23'} <class 'str'>


In [2]:
total = ""
for row in rows:
    total = total + row["age"]       # + on strings is concatenation
print(total, "<-", len(total), "characters")

23413529 <- 8 characters


Nobody wrote a wrong line: `+` did exactly what it means for the type it was
given. The fix is a conversion.

In [3]:
total = 0
for row in rows:
    total = total + int(row["age"])  # now + is addition
print(total)

128


## Why Python let it happen

Python is *dynamically typed*: the operator is chosen at run time from the
types of the operands. Mixing the two types is caught, but two strings are a
valid pair for `+`.

In [4]:
print(type("23"), type(23))
try:
    0 + "23"
except TypeError as e:
    print("mixed types are refused:", e)

<class 'str'> <class 'int'>
mixed types are refused: unsupported operand type(s) for +: 'int' and 'str'


## The same bug without a loop

A pandas column of text does the same thing when you ask for its sum, and it
does it silently.

In [5]:
import pandas as pd

ages = pd.Series([row["age"] for row in rows])
print(repr(ages.sum()), "   dtype:", ages.dtype)
print(ages.astype(int).sum())

'23413529'    dtype: object
128


The debugging habit this teaches: when a result looks wrong, print the **type**
of the value, not just the value.

**What we used:** the type system, dynamic typing and operator semantics
(session 4, sections 4 and 5); debugging (section 12).